<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_01_exploratory_data_analysis/exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 1 folder
%cd chapter_01_exploratory_data_analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 140 (delta 84), reused 72 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 490.47 KiB | 13.62 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_01_exploratory_data_analysis


# Chapter 01 Exercises: Exploratory Data Analysis

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 1: Exploratory Data Analysis  
> Authors: Peter Bruce, Andrew Bruce, Peter Gedeck

## Goals

In this notebook, I will:
- Practise Chapter 1 concepts
- Apply EDA using Python
- Interpret statistical measures
- Identify outliers
- Understand distributions
- Build intuition for data exploration

### Topics covered
- Mean vs median
- Weighted mean
- Percentiles
- Variability
- Boxplots
- Histograms
- Frequency tables
- Correlation
- Missing values
- Outlier detection

---

## 📋 Instructions

- Attempt each exercise before viewing the solution
- Use `utils/notebook_setup.py` for consistent imports
- Save your work frequently and commit progress to GitHub
- Solutions are provided in collapsed cells—expand only after attempting the problem

---



## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import trim_mean, shapiro, probplot, linregress
    from statsmodels.robust.scale import mad
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

print("Setup complete")
```
</details>


## Load Dataset

<details>
<summary>Click to view solution</summary>

```python
# Load dataset
state = pd.read_csv("../datasets/raw/state.csv")
state.head()
```
</details>


---

## Exercise 1: Data Types and Rectangular Data

### Problem 1.1
Load the `state.csv` dataset. Identify and list the data type (numeric continuous, numeric discrete, categorical, binary, ordinal) for each column. Convert any columns to more appropriate pandas dtypes.

<details>
<summary>Click to view solution</summary>

```python
from utils.notebook_setup import *
import os

# Load data
data_path = '../datasets/raw/state.csv'
if os.path.exists(data_path):
    state = pd.read_csv(data_path)
else:
    url = 'https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/state.csv'
    state = pd.read_csv(url)

# Inspect types
print("Original dtypes:")
print(state.dtypes)

# Convert to appropriate types
state['State'] = state['State'].astype('category')
state['Abbreviation'] = state['Abbreviation'].astype('category')

print("\nUpdated dtypes:")
print(state.dtypes)

# Classification:
# - Population: numeric continuous (int but represents continuous quantity)
# - Murder.rate: numeric continuous (float)
# - State, Abbreviation: categorical (nominal)
```
</details>


### Problem 1.2
Create a new column `Population_Millions` that divides `Population` by 1,000,000. Then create a binary column `HighPopulation` that is `1` if population > median, `0` otherwise.

<details>
<summary>Click to view solution</summary>

```python
# Create new columns
state['Population_Millions'] = state['Population'] / 1_000_000
median_pop = state['Population'].median()
state['HighPopulation'] = (state['Population'] > median_pop).astype(int)

# Display
state[['State', 'Population', 'Population_Millions', 'HighPopulation']].head(10)
```
</details>


---

## Exercise 2: Mean vs Median

### Problem 2.1

Calculate:
- Mean population
- Median population

Then answer:
1. Which measure is larger?
2. Why?
3. Which one would you trust more?

<details>
<summary>Click to view solution</summary>

```python
population_mean = state["Population"].mean()
population_median = state["Population"].median()

print(f"Mean population: {population_mean:,.2f}")
print(f"Median population: {population_median:,.2f}")

# Interpretation:
# If mean > median, distribution is right-skewed (a few large states pull mean upward)
# Median better represents a "typical" state population
```
</details>


### Problem 2.2

Calculate a 10% trimmed mean for population. How does it compare with the normal mean?

<details>
<summary>Click to view solution</summary>

```python
from scipy.stats import trim_mean

trimmed = trim_mean(state["Population"], 0.10)

print(f"Trimmed mean (10%): {trimmed:,.2f}")
print(f"Regular mean: {state['Population'].mean():,.2f}")

# Reflection:
# Trimmed mean is useful when data has outliers or is skewed
# It removes extreme values from both ends before calculating
# More robust than regular mean, less extreme than median
```
</details>

---

## Exercise 3: Estimates of Location

### Problem 3.1
For the `Murder.rate` column, compute:
- Mean
- Median
- 10% trimmed mean
- Weighted mean (weighted by `Population`)

Interpret the differences.

<details>
<summary>Click to view solution</summary>

```python
from scipy.stats import trim_mean

murder = state['Murder.rate']
pop = state['Population']

mean_val = murder.mean()
median_val = murder.median()
trimmed_val = trim_mean(murder, 0.10)
weighted_val = np.average(murder, weights=pop)

print(f"Mean: {mean_val:.2f}")
print(f"Median: {median_val:.2f}")
print(f"10% Trimmed Mean: {trimmed_val:.2f}")
print(f"Population-Weighted Mean: {weighted_val:.2f}")

# Interpretation:
# If mean > median, distribution is right-skewed (a few states have high murder rates)
# Weighted mean accounts for population size—more representative of national average
```
</details>


### Problem 3.2
Which measure of location is most appropriate for describing "typical" state murder rate? Justify with a visualization.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(8, 5))
sns.histplot(data=state, x='Murder.rate', bins=12, kde=True, color='skyblue')
plt.axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
plt.axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.2f}')
plt.axvline(trimmed_val, color='orange', linestyle='--', label=f'Trimmed: {trimmed_val:.2f}')
plt.xlabel('Murder Rate (per 100,000)')
plt.ylabel('Frequency')
plt.title('Distribution of State Murder Rates')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Conclusion:
# If distribution is roughly symmetric with no extreme outliers, mean is fine
# If skewed or outliers present, median or trimmed mean is more robust
# For murder rates, median is often preferred due to potential outliers
```
</details>


---

## Exercise 4: Percentiles

### Task
Calculate:
- 25th percentile
- 50th percentile
- 75th percentile
- 90th percentile

<details>
<summary>Click to view solution</summary>

```python
percentiles = np.percentile(
    state["Population"],
    [25, 50, 75, 90]
)

for p, val in zip([25, 50, 75, 90], percentiles):
    print(f"{p}th percentile: {val:,.0f}")

# Questions:
# 1. 90th percentile means 90% of states have population below this value
# 2. Percentiles are useful for understanding spread and identifying outliers
```
</details>


---

## Exercise 5: Variability

### Problem 5.1
For `Population`, compute:
- Variance
- Standard deviation
- Interquartile range (IQR)
- Median absolute deviation (MAD)

Which metric is most robust to outliers? Explain.

<details>
<summary>Click to view solution</summary>

```python
from statsmodels.robust.scale import mad

pop = state['Population']

variance = pop.var()
std = pop.std()
q75, q25 = np.percentile(pop, [75, 25])
iqr = q75 - q25
mad_val = mad(pop)

print(f"Variance: {variance:,.2f}")
print(f"Standard Deviation: {std:,.2f}")
print(f"IQR: {iqr:,.2f}")
print(f"MAD: {mad_val:,.2f}")

# Interpretation:
# Which metric is easiest to understand? Standard deviation (same units as data)
# SD and variance are sensitive to outliers (based on squared deviations from mean)
# IQR and MAD are robust—they use percentiles/medians, not means
# MAD is most robust for highly skewed data like population
```
</details>


### Problem 5.2
Identify outliers in `Population` using the 1.5×IQR rule. List the outlier states and visualize them on a boxplot.

<details>
<summary>Click to view solution</summary>

```python
# Identify outliers
lower_bound = q25 - 1.5 * iqr
upper_bound = q75 + 1.5 * iqr
outliers = state[(pop < lower_bound) | (pop > upper_bound)]

print(f"Outlier states: {outliers['State'].tolist()}")

# Boxplot
plt.figure(figsize=(10, 4))
sns.boxplot(x=state['Population'])
plt.title('Population Boxplot with Outliers')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>


---

## Exercise 6.1: Histogram

### Task
Visualise population distribution and assess skewness.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(10, 6))
sns.histplot(state["Population"], bins=20, kde=True)
plt.title("Population Distribution")
plt.xlabel("Population")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Questions:
# 1. Distribution is right-skewed (long tail to the right)
# 2. Not symmetric—most states clustered at lower populations
# 3. Yes, large states like California and Texas are potential outliers
```
</details>


## Exercise 6.2: Boxplot

### Task
Identify possible outliers using boxplot.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(10, 4))
sns.boxplot(x=state["Population"])
plt.title("Population Boxplot")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Reflection:
# Which states might be outliers? California, Texas, New York, Florida
# Why? Their populations are far above the upper whisker
```
</details>


---

## Exercise 7: Exploring Data Distribution

### Problem 7.1
Create a density plot and a QQ-plot for `Murder.rate`. Assess whether the data is approximately normally distributed.

<details>
<summary>Click to view solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Density plot
sns.kdeplot(data=state, x='Murder.rate', fill=True, ax=axes[0], color='skyblue')
axes[0].set_title('Density Plot: Murder Rate')
axes[0].set_xlabel('Murder Rate (per 100,000)')
axes[0].grid(alpha=0.3)

# QQ-plot
stats.probplot(state['Murder.rate'], dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot: Murder Rate vs Normal')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Interpretation:
# If QQ-plot shows S-shape or curvature, data is not normal
# Murder rates often right-skewed → likely not perfectly normal
```
</details>

### Problem 7.2
Compute the 5th, 25th, 50th, 75th, and 95th percentiles of `Murder.rate`. Use these to describe the spread and tails of the distribution.

<details>
<summary>Click to view solution</summary>

```python
percentiles = [0.05, 0.25, 0.5, 0.75, 0.95]
values = state['Murder.rate'].quantile(percentiles)

print("Murder Rate Percentiles:")
for p, v in zip(percentiles, values):
    print(f"  {int(p*100)}th percentile: {v:.2f}")

# Interpretation:
# IQR = Q3 - Q1 → middle 50% spread
# Tail comparison: (Q3-Q2) vs (Q2-Q1) indicates skew direction
# 95th - 5th → overall range excluding most extreme values
print(f"\nIQR: {values.iloc[3] - values.iloc[1]:.2f}")
print(f"Overall range (95th - 5th): {values.iloc[4] - values.iloc[0]:.2f}")
```
</details>


## Exercise 7.3: Frequency Tables

### Task
Create a frequency table for state regions.

<details>
<summary>Click to view solution</summary>

```python
# Create Region mapping from Abbreviation
region_map = {
    'AL': 'South', 'AK': 'West', 'AZ': 'West', 'AR': 'South', 'CA': 'West',
    'CO': 'West', 'CT': 'Northeast', 'DE': 'Northeast', 'FL': 'South',
    'GA': 'South', 'HI': 'West', 'ID': 'West', 'IL': 'Midwest', 'IN': 'Midwest',
    'IA': 'Midwest', 'KS': 'Midwest', 'KY': 'South', 'LA': 'South', 'ME': 'Northeast',
    'MD': 'South', 'MA': 'Northeast', 'MI': 'Midwest', 'MN': 'Midwest', 'MS': 'South',
    'MO': 'Midwest', 'MT': 'West', 'NE': 'Midwest', 'NV': 'West', 'NH': 'Northeast',
    'NJ': 'Northeast', 'NM': 'West', 'NY': 'Northeast', 'NC': 'South', 'ND': 'Midwest',
    'OH': 'Midwest', 'OK': 'South', 'OR': 'West', 'PA': 'Northeast', 'RI': 'Northeast',
    'SC': 'South', 'SD': 'Midwest', 'TN': 'South', 'TX': 'South', 'UT': 'West',
    'VT': 'Northeast', 'VA': 'South', 'WA': 'West', 'WV': 'South', 'WI': 'Midwest', 'WY': 'West'
}
state['Region'] = state['Abbreviation'].map(region_map)

# Frequency table
print(state["Region"].value_counts())

# Questions:
# 1. Which region appears most?
# 2. Does frequency always mean importance? No—consider proportions and context
```
</details>


### Problem 8.1

Create a new categorical variable `MurderLevel` with categories: "Low" (<25th percentile), "Medium" (25th–75th), "High" (>75th). Plot a bar chart showing the count of states in each level.

<details>
<summary>Click to view solution</summary>

```python
# Create MurderLevel
state['MurderLevel'] = pd.qcut(
    state['Murder.rate'],
    q=[0, 0.25, 0.75, 1],
    labels=['Low', 'Medium', 'High']
)

# Bar chart
counts = state['MurderLevel'].value_counts().sort_index()
plt.figure(figsize=(6, 5))
sns.barplot(x=counts.index, y=counts.values, palette='viridis')
plt.xlabel('Murder Level')
plt.ylabel('Number of States')
plt.title('States by Murder Rate Level')
for i, v in enumerate(counts.values):
    plt.text(i, v + 0.5, str(v), ha='center')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>


### Problem 8.2
Compute the mode of `MurderLevel`. What does this tell you about the distribution of murder rates across states?

<details>
<summary>Click to view solution</summary>

```python
mode_level = state['MurderLevel'].mode()[0]
mode_count = (state['MurderLevel'] == mode_level).sum()

print(f"Mode: {mode_level} ({mode_count} states, {mode_count/len(state)*100:.1f}%)")

# Interpretation:
# If mode is "Medium", most states have moderate murder rates
# This aligns with expectation—extremes are less common
```
</details>


### Problen 8.3

Compute both Pearson and Spearman correlation between `Population` and `Murder.rate`. Interpret the difference.

<details>
<summary>Click to view solution</summary>

```python
pearson = state['Population'].corr(state['Murder.rate'], method='pearson')
spearman = state['Population'].corr(state['Murder.rate'], method='spearman')

print(f"Pearson correlation: {pearson:.3f}")
print(f"Spearman correlation: {spearman:.3f}")

# Interpretation:
# Pearson measures linear relationship
# Spearman measures monotonic (rank-based) relationship
# If |Pearson| < |Spearman|, relationship may be monotonic but not linear
# If both near 0, little association between variables
```
</details>


---

## Exercise 9: Frequency Tables

### Task
Create a frequency table for state regions.

<details>
<summary>Click to view solution</summary>

```python
# Create Region from Abbreviation
region_map = {
    'AL': 'South', 'AK': 'West', 'AZ': 'West', 'AR': 'South', 'CA': 'West',
    'CO': 'West', 'CT': 'Northeast', 'DE': 'Northeast', 'FL': 'South',
    'GA': 'South', 'HI': 'West', 'ID': 'West', 'IL': 'Midwest', 'IN': 'Midwest',
    'IA': 'Midwest', 'KS': 'Midwest', 'KY': 'South', 'LA': 'South', 'ME': 'Northeast',
    'MD': 'South', 'MA': 'Northeast', 'MI': 'Midwest', 'MN': 'Midwest', 'MS': 'South',
    'MO': 'Midwest', 'MT': 'West', 'NE': 'Midwest', 'NV': 'West', 'NH': 'Northeast',
    'NJ': 'Northeast', 'NM': 'West', 'NY': 'Northeast', 'NC': 'South', 'ND': 'Midwest',
    'OH': 'Midwest', 'OK': 'South', 'OR': 'West', 'PA': 'Northeast', 'RI': 'Northeast',
    'SC': 'South', 'SD': 'Midwest', 'TN': 'South', 'TX': 'South', 'UT': 'West',
    'VT': 'Northeast', 'VA': 'South', 'WA': 'West', 'WV': 'South', 'WI': 'Midwest', 'WY': 'West'
}
state['Region'] = state['Abbreviation'].map(region_map)

# Frequency table
region_counts = state['Region'].value_counts()
print(region_counts)

# Questions:
# 1. Which region appears most?
# 2. Does frequency always mean importance? No—consider proportions and context
```
</details>


---

## Exercise 10: Correlation

### Problem 10.1
Compute Pearson and Spearman correlation between `Population` and `Murder.rate`.

<details>
<summary>Click to view solution</summary>

```python
pearson = state['Population'].corr(state['Murder.rate'], method='pearson')
spearman = state['Population'].corr(state['Murder.rate'], method='spearman')

print(f"Pearson correlation: {pearson:.3f}")
print(f"Spearman correlation: {spearman:.3f}")

# Interpretation:
# If |Pearson| < |Spearman|, relationship may be monotonic but not linear
# If both near 0, little association
```
</details>


### Problem 10.2
Create a scatterplot of `Population` (x-axis, in millions) vs. `Murder.rate` (y-axis). Add a regression line and annotate the correlation coefficient.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(8, 6))
sns.regplot(data=state, x='Population_Millions', y='Murder.rate',
            scatter_kws={'alpha': 0.7, 's': 50}, line_kws={'color': 'red'})

# Annotate correlation
slope, intercept, r, p, _ = linregress(state['Population_Millions'], state['Murder.rate'])
plt.text(0.1, 8, f'r = {pearson:.3f}\np = {p:.3f}', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.xlabel('Population (millions)')
plt.ylabel('Murder Rate (per 100,000)')
plt.title('Population vs. Murder Rate by State')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Regression: MurderRate = {intercept:.2f} + {slope:.4f} × Population(millions)")
```
</details>



---

## Exercise 11: Exploring Two or More Variables

### Problem 11.1
Create a boxplot of `Murder.rate` by `Region`.

<details>
<summary>Click to view solution</summary>

```python
# Summary stats by region
print("Murder Rate by Region:")
print(state.groupby('Region')['Murder.rate'].agg(['mean', 'median', 'std', 'count']))

# Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(data=state, x='Region', y='Murder.rate',
            order=sorted(state['Region'].unique()))
plt.xlabel('Region')
plt.ylabel('Murder Rate (per 100,000)')
plt.title('Murder Rate Distribution by Region')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Interpretation:
# Southern states show higher median murder rates and greater variability
# Northeast has lowest median and tightest IQR
# Suggests regional factors may influence murder rates
```
</details>


### Problem 11.2

Using the `telecom.csv` dataset, create a hexagonal binning plot of `DailyMins` vs. `DailyCalls`. Interpret the pattern.

<details>
<summary>Click to view solution</summary>

```python
# Load telecom data
def load_book_data(filename):
    """Helper to load book datasets"""
    path = f'../datasets/raw/{filename}'
    if os.path.exists(path):
        return pd.read_csv(path)
    else:
        url = f'https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/{filename}'
        return pd.read_csv(url)

telecom = load_book_data('telecom.csv')

# Hexbin plot
plt.figure(figsize=(8, 6))
plt.hexbin(telecom['DailyCalls'], telecom['DailyMins'],
           gridsize=30, cmap='viridis', mincnt=1)
plt.colorbar(label='Count')
plt.xlabel('Daily Calls')
plt.ylabel('Daily Minutes')
plt.title('Hexagonal Binning: Calls vs. Minutes')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Interpretation:
# Dense region in lower-left → most customers have few calls/minutes
# Sparse upper-right → heavy users are rare
# Positive association expected: more calls → more minutes
```
</details>


### Problem 11.3

Create a contingency table of `Churn` (binary) vs. `IntlPlan` (Yes/No). Compute row percentages and visualize with a stacked bar chart.

<details>
<summary>Click to view solution</summary>

```python
# Contingency table with row percentages
contingency = pd.crosstab(telecom['Churn'], telecom['IntlPlan'], normalize='index') * 100

print("Row Percentages (Churn by IntlPlan):")
print(contingency.round(1))

# Stacked bar chart
contingency.plot(kind='bar', stacked=True, figsize=(6, 5), colormap='Set2')
plt.ylabel('Percentage (%)')
plt.xlabel('Churn')
plt.title('Churn Rate by International Plan')
plt.legend(title='Intl Plan')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Interpretation:
# If churn % is higher for "Yes" IntlPlan, plan may correlate with churn
# Useful for targeting retention efforts
```
</details>


---

## Mini Experiment: Remove Outliers

Remove the largest 5 states by population and compare mean vs median.

<details>
<summary>Click to view solution</summary>

```python
cleaned = state.sort_values(by="Population").iloc[:-5]

print("Original Mean:", state["Population"].mean())
print("New Mean:", cleaned["Population"].mean())
print("\nOriginal Median:", state["Population"].median())
print("New Median:", cleaned["Population"].median())

# Insight:
# Mean changed more than median because it's sensitive to extreme values
# Median remains relatively stable when outliers are removed
```
</details>


---

## ML Relevance Exercise

Why is EDA important before machine learning?

Think about:
- Bad data
- Outliers
- Skewness
- Missing values
- Feature quality

<details>
<summary>Click to view solution</summary>

```python
# EDA is critical because:
# 1. It reveals data quality issues (missing values, outliers)
# 2. It informs feature engineering (scaling, encoding decisions)
# 3. It prevents modeling assumptions that contradict data reality
# 4. It builds intuition about relationships before committing to a model
# 5. It helps identify potential data leakage and biases
```
</details>


---

## Interview Questions

Answer these common interview questions:

1. **Difference between mean and median?**
2. **When would median be preferred?**
3. **How do you detect outliers?**
4. **Why is EDA important?**
5. **What does correlation mean?**

<details>
<summary>Click to view solution</summary>

```python
answers = """
1. Difference between mean and median?
   - Mean: arithmetic average of all values
   - Median: middle value when data is sorted
   - Mean is sensitive to outliers; median is robust

2. When would median be preferred?
   - When data is skewed (income, house prices)
   - When outliers are present
   - When describing "typical" value in non-normal distributions
   - Example: Median home price vs. average home price

3. How do you detect outliers?
   - IQR method: values outside [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
   - Z-score: |z| > 3 (for normal-ish distributions)
   - Visual methods: boxplots, scatter plots
   - Domain knowledge: values that don't make physical sense
   - ML methods: Isolation Forest, DBSCAN

4. Why is EDA important?
   - Understand data structure and quality
   - Identify patterns, relationships, and anomalies
   - Guide feature engineering and model selection
   - Detect data issues before they impact models
   - Build intuition that informs analysis decisions

5. What does correlation mean?
   - Measures strength and direction of linear relationship
   - Ranges from -1 (perfect negative) to +1 (perfect positive)
   - 0 means no linear relationship
   - Correlation ≠ causation!
   - Always visualize relationships, don't rely on correlation alone
"""
print(answers)
```
</details>


Choose one numerical column and perform complete EDA:
1. Summary statistics
2. Histogram
3. Boxplot
4. Percentiles
5. Outlier analysis
6. Interpretation

<details>
<summary>Click to view solution</summary>

```python
def complete_eda(df, col):
    """Complete EDA for a single numeric column"""
    series = df[col]
    
    print(f"{'='*50}")
    print(f"Complete EDA: {col}")
    print(f"{'='*50}")
    
    # 1. Summary statistics
    print(f"\n1. Summary Statistics:")
    print(f"   Mean: {series.mean():.2f}")
    print(f"   Median: {series.median():.2f}")
    print(f"   Std Dev: {series.std():.2f}")
    print(f"   Min: {series.min():.2f}")
    print(f"   Max: {series.max():.2f}")
    print(f"   Skewness: {series.skew():.2f}")
    print(f"   Kurtosis: {series.kurtosis():.2f}")
    
    # 2 & 3. Histogram and Boxplot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    sns.histplot(series, bins=20, kde=True, ax=axes[0], color='skyblue')
    axes[0].axvline(series.mean(), color='red', linestyle='--', label=f'Mean: {series.mean():.2f}')
    axes[0].axvline(series.median(), color='green', linestyle='--', label=f'Median: {series.median():.2f}')
    axes[0].set_title(f'Histogram: {col}')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    sns.boxplot(x=series, ax=axes[1], color='lightblue')
    axes[1].set_title(f'Boxplot: {col}')
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 4. Percentiles
    print(f"\n4. Percentiles:")
    for p in [5, 10, 25, 50, 75, 90, 95]:
        print(f"   {p}th: {np.percentile(series, p):.2f}")
    
    # 5. Outlier analysis
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    
    print(f"\n5. Outlier Analysis (IQR method):")
    print(f"   Lower bound: {lower_bound:.2f}")
    print(f"   Upper bound: {upper_bound:.2f}")
    print(f"   Outliers detected: {len(outliers)} ({len(outliers)/len(series)*100:.1f}%)")
    if len(outliers) > 0:
        print(f"   Outlier values: {outliers.tolist()}")
    
    # 6. Interpretation
    print(f"\n6. Interpretation:")
    skew = series.skew()
    if skew > 1:
        print(f"   Highly right-skewed (skewness={skew:.2f})")
        print(f"   Recommendation: Consider log transformation")
        print(f"   Use median/IQR over mean/SD for robust statistics")
    elif skew < -1:
        print(f"   Highly left-skewed (skewness={skew:.2f})")
        print(f"   Recommendation: Consider reflection + log transformation")
    elif abs(skew) > 0.5:
        print(f"   Moderately skewed (skewness={skew:.2f})")
        print(f"   Recommendation: Trimmed mean or winsorization may help")
    else:
        print(f"   Approximately symmetric (skewness={skew:.2f})")
        print(f"   Recommendation: Mean and SD are appropriate")

# Example usage
complete_eda(state, 'Murder.rate')
print("\n" + "="*50 + "\n")
complete_eda(state, 'Population')
```
</details>

## Answer in your own words:
1. Why is EDA a critical first step before building any predictive model?
2. When would you prefer median over mean? Give a real-world example from this chapter.
3. How might correlation mislead if used without visualization?

<details>
<summary>Click to view solution</summary>

```python
reflection = """
1. EDA is critical before modeling because:
   - Reveals data quality issues (missing values, outliers, wrong types)
   - Informs feature engineering decisions (scaling, encoding)
   - Prevents modeling assumptions that contradict data reality
   - Builds intuition about relationships before model selection
   - Can save hours of debugging by catching issues early

2. Prefer median when:
   - Data is skewed or has outliers
   - Example from chapter: State population data
     California and Texas pull mean upward significantly
     Median better represents a "typical" state's population
   - Other real-world examples: income, house prices, website load times

3. Correlation without visualization can mislead because:
   - Only captures linear relationships (non-linear = 0 correlation)
   - Outliers can dramatically inflate or deflate correlation
   - Anscombe's quartet: 4 datasets with identical correlation
     but completely different scatterplots
   - Always plot your data! Correlation is a summary, not a replacement
     for visualization
"""
print(reflection)
```
</details>



---

## 🎯 Bonus Challenges

### Bonus Challenge 1: Replicate Book Figure 1-4 (Density Plot)
Recreate the density plot of murder rates with histogram overlay.

<details>
<summary>Click to view solution</summary>

```python
plt.figure(figsize=(10, 6))

# Histogram with density overlay
sns.histplot(state['Murder.rate'], bins=12, stat='density',
             alpha=0.3, color='skyblue', label='Histogram')
sns.kdeplot(state['Murder.rate'], color='darkblue', linewidth=2, label='Density')

plt.xlabel('Murder Rate (per 100,000)')
plt.ylabel('Density')
plt.title('Distribution of State Murder Rates')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>



### Bonus Challenge 2: Automated EDA Function
Write a function `quick_eda(df, col)` that:
- Prints location/variability stats for numeric columns
- Prints mode/proportions for categorical columns
- Generates an appropriate plot (histogram, bar chart, or boxplot)

<details>
<summary>Click to view solution</summary>

```python
def quick_eda(df, col):
    """Quick EDA for a single column"""
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"Numeric Column: {col}")
        print(f"  Mean: {df[col].mean():.2f}")
        print(f"  Median: {df[col].median():.2f}")
        print(f"  Std Dev: {df[col].std():.2f}")
        print(f"  IQR: {df[col].quantile(0.75) - df[col].quantile(0.25):.2f}")
        print(f"  Skewness: {df[col].skew():.2f}")
        print(f"  Missing: {df[col].isnull().sum()}")
        
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        sns.histplot(df[col], kde=True)
        plt.title(f'Distribution of {col}')
        plt.subplot(1, 2, 2)
        sns.boxplot(x=df[col])
        plt.title(f'Boxplot of {col}')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Categorical Column: {col}")
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Mode: {df[col].mode().values[0] if not df[col].mode().empty else 'N/A'}")
        print(f"  Missing: {df[col].isnull().sum()}")
        print("\n  Value Proportions:")
        print(df[col].value_counts(normalize=True).round(3))
        
        plt.figure(figsize=(8, 4))
        df[col].value_counts().plot(kind='bar', color='skyblue')
        plt.title(f'Counts of {col}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

# Example usage:
# quick_eda(state, 'Murder.rate')
# quick_eda(state, 'Region')
```
</details>



---

## Additional Practice: Missing Data Audit

Write a reusable function `audit_missing(df)` that reports and visualizes missing values.

<details>
<summary>Click to view solution</summary>

```python
def audit_missing(df):
    """Audit missing values in dataframe"""
    missing = df.isnull().sum()
    pct_missing = (missing / len(df)) * 100
    report = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': pct_missing.round(2)
    }).query('missing_count > 0')
    
    if not report.empty:
        print("⚠️ Missing values found:")
        display(report)
        
        # Visualization
        report['missing_pct'].plot(kind='barh', figsize=(8, 4), color='salmon')
        plt.xlabel('Percent Missing')
        plt.title('Missing Data Audit')
        plt.tight_layout()
        plt.show()
        
        # Strategy suggestions
        print("\n💡 Suggested strategies:")
        for col in report.index:
            if pd.api.types.is_numeric_dtype(df[col]):
                print(f"  • {col}: Consider median imputation (robust to outliers)")
            else:
                print(f"  • {col}: Consider mode imputation or 'Unknown' category")
    else:
        print("✅ No missing values found!")
    
    return report

# Test
audit_missing(state)
```
</details>


---

## Additional Practice: Outlier Investigation Protocol

Implement outlier detection using multiple methods and compare results.

<details>
<summary>Click to view solution</summary>

```python
from sklearn.ensemble import IsolationForest

def detect_outliers_comprehensive(series):
    """Detect outliers using multiple methods"""
    s = series.dropna()
    results = {}
    
    # Method 1: IQR
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    results['IQR'] = set(s[(s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)].index)
    
    # Method 2: Z-score
    z = np.abs((s - s.mean()) / s.std())
    results['Z-score'] = set(s[z > 3].index)
    
    # Method 3: Isolation Forest
    iso = IsolationForest(contamination=0.05, random_state=42)
    preds = iso.fit_predict(s.values.reshape(-1, 1))
    results['IsolationForest'] = set(s[preds == -1].index)
    
    # Report
    print(f"Outlier Detection Results for '{series.name}':")
    print("-" * 40)
    for method, idx_set in results.items():
        print(f"  {method:20s}: {len(idx_set):3d} outliers ({len(idx_set)/len(s)*100:.1f}%)")
    
    # Common outliers
    common = set.intersection(*results.values())
    print(f"\n  Flagged by ALL methods: {len(common)} records")
    
    return results

# Test
outlier_results = detect_outliers_comprehensive(state['Population'])
```
</details>


---

## Additional Practice: Correlation Explorer

Compare Pearson and Spearman correlations and highlight nonlinear relationships.

<details>
<summary>Click to view solution</summary>

```python
def correlation_explorer(df, columns):
    """Explore and compare correlation methods"""
    pearson = df[columns].corr(method='pearson')
    spearman = df[columns].corr(method='spearman')
    diff = np.abs(pearson - spearman)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.heatmap(pearson, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], center=0)
    axes[0].set_title('Pearson Correlation')
    
    sns.heatmap(spearman, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], center=0)
    axes[1].set_title('Spearman Correlation')
    
    sns.heatmap(diff, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[2])
    axes[2].set_title('|Pearson − Spearman| Difference')
    
    plt.tight_layout()
    plt.show()
    
    # Report large differences
    for i in range(len(columns)):
        for j in range(i+1, len(columns)):
            if diff.iloc[i, j] > 0.2:
                print(f"⚠️ {columns[i]} vs {columns[j]}: "
                      f"Pearson={pearson.iloc[i,j]:.3f}, "
                      f"Spearman={spearman.iloc[i,j]:.3f} "
                      f"(Possible nonlinear relationship)")
    
    return {'pearson': pearson, 'spearman': spearman, 'diff': diff}

# Test
numeric_cols = state.select_dtypes('number').columns.tolist()
correlation_explorer(state, numeric_cols)
```
</details>

---



### Synthesis & Critical Thinking

Choose one numeric and one categorical variable from `state.csv`. Design a small EDA workflow:
1. Summarize each variable (location + variability for numeric; mode + proportions for categorical)
2. Visualize their relationship (boxplot, violin plot, or grouped bar chart)
3. Write 2–3 sentences interpreting what you learned

<details>
<summary>Click to view solution</summary>

```python
# Example: Murder.rate by Region

# 1. Summary statistics
print("Numeric Variable (Murder.rate):")
print(f"  Mean: {state['Murder.rate'].mean():.2f}")
print(f"  Median: {state['Murder.rate'].median():.2f}")
print(f"  IQR: {state['Murder.rate'].quantile(0.75) - state['Murder.rate'].quantile(0.25):.2f}")

print("\nCategorical Variable (Region):")
region_props = state['Region'].value_counts(normalize=True)
print(region_props)

# 2. Visualization
plt.figure(figsize=(10, 6))
sns.boxplot(data=state, x='Region', y='Murder.rate',
            order=sorted(state['Region'].unique()), palette='Set2')
sns.stripplot(data=state, x='Region', y='Murder.rate',
              order=sorted(state['Region'].unique()), color='black', alpha=0.3, size=4)
plt.xlabel('Region')
plt.ylabel('Murder Rate (per 100,000)')
plt.title('Murder Rate Distribution by Region')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Interpretation
print("\nInterpretation:")
print("Southern states show higher median murder rates and greater variability.")
print("Northeast has lowest median and tightest IQR, suggesting more consistency.")
print("Suggests regional factors (policy, demographics, economics) may influence murder rates.")
```
</details>

---

## Progress Checklist

- [ ] Completed data type identification
- [ ] Practised mean vs median
- [ ] Calculated trimmed and weighted means
- [ ] Practised percentiles
- [ ] Understood variability (variance, SD, IQR, MAD)
- [ ] Created histograms and density plots
- [ ] Used boxplots for outlier detection
- [ ] Created frequency tables
- [ ] Analysed correlations (Pearson and Spearman)
- [ ] Explored bivariate relationships
- [ ] Completed mini experiment (outlier removal)
- [ ] Finished interview questions
- [ ] Completed challenge exercise
- [ ] Ready for Chapter 2

---
